# HgFinance Common QLoRA Training

This notebook is the first shared HgFinance behavior/finance adapter smoke test.

Common prepared dataset
→ Qwen/Qwen2.5-14B-Instruct
→ 4-bit NF4 QLoRA
→ hgfinance-common-v1

This is a common-only adapter, not a department adapter. The prepared Common
train and validation files are used exactly as supplied. No re-split, merge,
sampling, oversampling, or domain dataset is used.

The adapter is intended for later AWQ + LoRA compatibility and benchmark
validation. External-50, Internal-v1, and Internal-v2 / EmployeeReasoning
remain separate held-out promotion benchmarks.

In [1]:
from google.colab import drive
from pathlib import Path
import zipfile
import shutil

# 1) Drive mount
drive.mount("/content/drive")

# 2) 네가 방금 올린 ZIP
DRIVE_ZIP = Path("/content/drive/MyDrive/hgfinance_colab_training.zip")

if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"ZIP not found: {DRIVE_ZIP}")

# 3) Colab runtime repo 생성
REPO_ROOT = Path("/content/multi_agent")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

REPO_ROOT.mkdir(parents=True, exist_ok=True)

# 4) 압축 해제
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(REPO_ROOT)

# 5) 필수 파일 확인
checks = {
    "training": REPO_ROOT / "training",
    "qlora_script": REPO_ROOT / "scripts/qlora/train_specialist_qlora.py",
    "common": REPO_ROOT / "hgfinance_common_training_v1",
    "benchmarks": REPO_ROOT / "benchmarks/quantization",
    "train": REPO_ROOT / "training_runs/hgfinance-common-v1/prepared/train.jsonl",
    "validation": REPO_ROOT / "training_runs/hgfinance-common-v1/prepared/validation.jsonl",
}

print("REPO_ROOT:", REPO_ROOT)
for name, path in checks.items():
    print(f"{name:12s}: {path.exists()}")

assert all(path.exists() for path in checks.values())

print("\n✅ Colab training payload ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
REPO_ROOT: /content/multi_agent
training    : True
qlora_script: True
common      : True
benchmarks  : True
train       : True
validation  : True

✅ Colab training payload ready


In [2]:
# 1. Runtime and GPU check
import os
import json
import math
import hashlib
import inspect
import shutil
import subprocess
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A Colab CUDA GPU is required for training.")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16_SUPPORTED = torch.cuda.is_bf16_supported()

print("GPU:", GPU_NAME)
print(f"VRAM: {VRAM_GIB:.2f} GiB")
print("BF16 supported:", BF16_SUPPORTED)

GPU: NVIDIA L4
VRAM: 22.03 GiB
BF16 supported: True


In [3]:
# 2. Repository path setup
from pathlib import Path
import os

REPO_ROOT = Path(
    os.environ.get("HGFINANCE_REPO_ROOT", "/content/multi_agent")
).resolve()

if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"Repository root does not exist: {REPO_ROOT}"
    )

os.chdir(REPO_ROOT)
print("Repository:", REPO_ROOT)

Repository: /content/multi_agent


In [4]:
# 3. Install dependencies and configure Common-only training
%pip install -q -r training/qlora/requirements-colab.txt

import importlib.metadata as importlib_metadata

for package_name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets"):
    try:
        print(f"{package_name}: {importlib_metadata.version(package_name)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package_name}: NOT INSTALLED")

import random
import numpy as np

BASE_MODEL = "Qwen/Qwen2.5-14B-Instruct"
ADAPTER_NAME = "hgfinance-common-v1"
ADAPTER_VERSION = "v1"
TRAINING_MODE = "common_only"

PREPARED_DIR = REPO_ROOT / "training_runs/hgfinance-common-v1/prepared"
PREPARED_TRAIN_PATH = REPO_ROOT / "training_runs/hgfinance-common-v1/prepared/train.jsonl"
PREPARED_VALIDATION_PATH = REPO_ROOT / "training_runs/hgfinance-common-v1/prepared/validation.jsonl"
OUTPUT_DIR = REPO_ROOT / "training_runs" / "hgfinance-common-v1"
BENCHMARK_ROOT = REPO_ROOT / "benchmarks" / "quantization"

MAX_LENGTH = 3072
NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
SEED = 66
OPTIMIZER = "paged_adamw_8bit"

QLORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
assert QLORA_R <= 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Repository:", REPO_ROOT)
print("Training mode:", TRAINING_MODE)
print("Adapter:", ADAPTER_NAME)
print("Train:", PREPARED_TRAIN_PATH)
print("Validation:", PREPARED_VALIDATION_PATH)

torch: 2.11.0+cu128
transformers: 4.57.6
peft: 0.19.1
accelerate: 1.14.0
bitsandbytes: 0.50.1
datasets: 4.0.0
Repository: /content/multi_agent
Training mode: common_only
Adapter: hgfinance-common-v1
Train: /content/multi_agent/training_runs/hgfinance-common-v1/prepared/train.jsonl
Validation: /content/multi_agent/training_runs/hgfinance-common-v1/prepared/validation.jsonl


## Training sequence length decision

The initial tokenizer-only audit at `MAX_LENGTH = 2048` found 3 over-limit examples; the maximum observed full sequence length was 2331 tokens. `MAX_LENGTH = 3072` was selected to preserve every current sample without truncation. The fail-closed audit remains required for future records.

This is a training sequence-length decision only. Production vLLM serving remains `max_model_len = 8192`; this notebook does not change production configuration.

## Common-only smoke-test epoch decision

This first Common-only adapter run is a 1-epoch smoke-test/baseline. The notebook was initially configured for 3 epochs, but the projected runtime on a Colab NVIDIA L4 was roughly 9 hours. One epoch is selected first so AWQ + LoRA quality can be evaluated before spending compute on a longer run. Longer 2–3 epoch runs may be tested later if evaluation shows they are needed; 1 epoch is not guaranteed to match 3-epoch quality.


In [5]:
# 4. Prepared JSONL integrity check
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl_records(path: Path) -> list[dict]:
    if not path.is_file():
        raise FileNotFoundError(path)
    records = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_number}: invalid JSON") from exc
            if not isinstance(record, dict):
                raise ValueError(f"{path}:{line_number}: record must be an object")
            records.append(record)
    if not records:
        raise ValueError(f"{path}: no records")
    return records

def validate_record(record: dict, source: str) -> None:
    if not record.get("id"):
        raise ValueError(f"{source}: missing id")
    messages = record.get("messages")
    if not isinstance(messages, list) or not messages:
        raise ValueError(f"{source}: messages must be a non-empty list")
    roles = [message.get("role") for message in messages]
    if roles[-1] != "assistant" or roles.count("assistant") != 1:
        raise ValueError(f"{source}: messages must end in exactly one assistant")
    if "system" not in roles or "user" not in roles:
        raise ValueError(f"{source}: system and user messages are required")
    for message in messages:
        if not isinstance(message.get("content"), str) or not message["content"].strip():
            raise ValueError(f"{source}: message content must be non-empty")

train_records = load_jsonl_records(PREPARED_TRAIN_PATH)
validation_records = load_jsonl_records(PREPARED_VALIDATION_PATH)
for index, record in enumerate(train_records):
    validate_record(record, f"train:{index}")
for index, record in enumerate(validation_records):
    validate_record(record, f"validation:{index}")

assert len(train_records) == 2545
assert len(validation_records) == 223

EXPECTED_COMMON_TRAIN_SHA256 = "5e52c0f5488b25a201b245b4fb6cae37d57c638490b7fd50ff205e70d05c328e"
EXPECTED_COMMON_VALIDATION_SHA256 = "371c77c3b569ac4d96d3e08148fba872daec134c088da81ea877c2c0e1ef104e"
common_source_dir = REPO_ROOT / "hgfinance_common_training_v1"
if (common_source_dir / "common_train.jsonl").is_file():
    assert sha256_file(common_source_dir / "common_train.jsonl") == EXPECTED_COMMON_TRAIN_SHA256
    assert sha256_file(common_source_dir / "common_validation.jsonl") == EXPECTED_COMMON_VALIDATION_SHA256

print(f"Prepared train records: {len(train_records)}")
print(f"Prepared validation records: {len(validation_records)}")
for record in (train_records[0], validation_records[0]):
    print({"id": record["id"], "category": record.get("category"), "roles": [m["role"] for m in record["messages"]]})

Prepared train records: 2545
Prepared validation records: 223
{'id': 'hgcommon-v2-00240', 'category': 'derivatives', 'roles': ['system', 'user', 'assistant']}
{'id': 'hgcommon-v2-01909', 'category': 'accounting', 'roles': ['system', 'user', 'assistant']}


In [6]:
# 5. Held-out contamination and split-safety checks
from training.specialist.schema import load_jsonl
from training.specialist.contamination import check_contamination, require_clean
from training.specialist.mixing import load_pool, preserve_pool_split

common_examples_train = load_jsonl(PREPARED_TRAIN_PATH, source_dataset="common")
common_examples_validation = load_jsonl(PREPARED_VALIDATION_PATH, source_dataset="common")

train_contamination = check_contamination(common_examples_train, BENCHMARK_ROOT)
validation_contamination = check_contamination(common_examples_validation, BENCHMARK_ROOT)
require_clean(train_contamination)
require_clean(validation_contamination)

common_pool = load_pool("common", PREPARED_TRAIN_PATH, PREPARED_VALIDATION_PATH)
assert len(preserve_pool_split(common_pool, "train")) == 2545
assert len(preserve_pool_split(common_pool, "validation")) == 223

BENCHMARK_CONTAMINATION = {
    "status": "PASS",
    "held_out": ["External-50", "Internal-v1", "Internal-v2 / EmployeeReasoning"],
    "train": train_contamination,
    "validation": validation_contamination,
}
print({
    "status": BENCHMARK_CONTAMINATION["status"],
    "benchmark_texts": train_contamination["benchmark_text_count"],
    "train_exact": train_contamination["exact_count"],
    "train_near": train_contamination["near_count"],
    "validation_exact": validation_contamination["exact_count"],
    "validation_near": validation_contamination["near_count"],
})

{'status': 'PASS', 'benchmark_texts': 500, 'train_exact': 0, 'train_near': 0, 'validation_exact': 0, 'validation_near': 0}


In [7]:
# 6. Resolve and pin the Hugging Face base revision before model loading
from huggingface_hub import model_info

REQUESTED_BASE_REVISION = os.environ.get("HF_BASE_REVISION") or None
MODEL_INFO = model_info(BASE_MODEL, revision=REQUESTED_BASE_REVISION)
RESOLVED_BASE_REVISION = MODEL_INFO.sha
if not RESOLVED_BASE_REVISION:
    raise RuntimeError("Could not resolve a Hugging Face commit SHA.")

print("Base model:", BASE_MODEL)
print("Requested revision:", REQUESTED_BASE_REVISION)
print("Resolved revision:", RESOLVED_BASE_REVISION)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Base model: Qwen/Qwen2.5-14B-Instruct
Requested revision: None
Resolved revision: cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8


In [8]:
# 7. Load tokenizer first and fail closed on full-sequence length
from transformers import AutoTokenizer
from training.specialist.schema import DatasetValidationError

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    revision=RESOLVED_BASE_REVISION,
    trust_remote_code=True,
)
if tokenizer.chat_template is None:
    raise RuntimeError("The Qwen tokenizer must provide a chat template.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def full_chat_token_ids(messages: list[dict]) -> list[int]:
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    encoded = tokenizer(rendered, add_special_tokens=False, truncation=False)
    return list(encoded["input_ids"])

def percentile(values: list[int], fraction: float) -> int:
    ordered = sorted(values)
    index = max(0, min(len(ordered) - 1, math.ceil(len(ordered) * fraction) - 1))
    return ordered[index]

all_records = [("train", record) for record in train_records]
all_records += [("validation", record) for record in validation_records]
lengths = []
over_limit = []
for split, record in all_records:
    token_length = len(full_chat_token_ids(record["messages"]))
    lengths.append(token_length)
    if token_length > MAX_LENGTH:
        over_limit.append({
            "split": split,
            "sample_id": record["id"],
            "full_token_length": token_length,
        })

TOKEN_LENGTH_AUDIT = {
    "example_count": len(lengths),
    "p50_token_length": percentile(lengths, 0.50),
    "p95_token_length": percentile(lengths, 0.95),
    "p99_token_length": percentile(lengths, 0.99),
    "max_token_length": max(lengths),
    "max_seq_length": MAX_LENGTH,
    "over_limit_count": len(over_limit),
    "over_limit_examples": over_limit,
}
if over_limit and len(over_limit) <= 10:
    print("Over-limit examples:")
    for item in over_limit:
        print(f"  split={item['split']} sample_id={item['sample_id']} full_token_length={item['full_token_length']}")
elif over_limit:
    print(f"Over-limit examples: {len(over_limit)} (details omitted)")

print(json.dumps({k: v for k, v in TOKEN_LENGTH_AUDIT.items() if k != "over_limit_examples"}, indent=2))
if over_limit:
    first = over_limit[0]
    raise DatasetValidationError(
        "Full sequence exceeds MAX_LENGTH; refusing model load. "
        f"sample_id={first['sample_id']} "
        f"full_token_length={first['full_token_length']} "
        f"max_seq_length={MAX_LENGTH}"
    )

{
  "example_count": 2768,
  "p50_token_length": 1053,
  "p95_token_length": 1517,
  "p99_token_length": 1745,
  "max_token_length": 2331,
  "max_seq_length": 3072,
  "over_limit_count": 0
}


In [9]:
# 8. Load the original Qwen base with 4-bit NF4 only after the audit passes
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

COMPUTE_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=RESOLVED_BASE_REVISION,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
print("Loaded:", BASE_MODEL)
print("Revision:", RESOLVED_BASE_REVISION)
print("Memory footprint:", f"{model.get_memory_footprint() / 1024**3:.2f} GiB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

model-00007-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/1.70G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-14B-Instruct
Revision: cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8
Memory footprint: 9.05 GiB


In [10]:
# 9. Explicit gradient checkpointing and QLoRA / PEFT configuration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

def prepare_kbit_model(model, qlora_config):
    prepare_parameters = inspect.signature(prepare_model_for_kbit_training).parameters
    kwargs = {}
    if "use_gradient_checkpointing" in prepare_parameters:
        kwargs["use_gradient_checkpointing"] = True
    if "gradient_checkpointing_kwargs" in prepare_parameters:
        kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    model = prepare_model_for_kbit_training(model, **kwargs)
    if "use_gradient_checkpointing" not in prepare_parameters:
        enable_parameters = inspect.signature(model.gradient_checkpointing_enable).parameters
        if "gradient_checkpointing_kwargs" in enable_parameters:
            model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        else:
            model.gradient_checkpointing_enable()
    return model

model = prepare_kbit_model(model, None)
peft_config = LoraConfig(
    r=QLORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
assert peft_config.r <= 32
model = get_peft_model(model, peft_config, adapter_name=ADAPTER_NAME)
model.print_trainable_parameters()
print(peft_config)

trainable params: 68,812,800 || all params: 14,838,846,464 || trainable%: 0.4637
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='Qwen/Qwen2.5-14B-Instruct', revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'q_proj', 'o_proj', 'k_proj', 'down_proj', 'v_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora

In [11]:
# 10. Build datasets and enforce assistant-only loss
from datasets import Dataset

train_dataset = Dataset.from_list(train_records)
validation_dataset = Dataset.from_list(validation_records)

class AssistantOnlyCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        rows = []
        for feature in features:
            messages = feature["messages"]
            full_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            prefix_text = self.tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=False,
                add_generation_prompt=True,
            )
            sample_id = feature.get("id", "unknown")
            full = self.tokenizer(
                full_text,
                add_special_tokens=False,
                truncation=False,
            )
            full_length = len(full["input_ids"])
            if full_length > self.max_length:
                raise DatasetValidationError(
                    "Assistant completion cannot be safely truncated: "
                    f"sample_id={sample_id} "
                    f"full_token_length={full_length} "
                    f"max_seq_length={self.max_length}"
                )
            prefix = self.tokenizer(
                prefix_text,
                add_special_tokens=False,
                truncation=False,
            )
            if full["input_ids"][:len(prefix["input_ids"])] != prefix["input_ids"]:
                raise DatasetValidationError(
                    "Qwen chat template prefix is not a full-sequence prefix"
                )
            if full_length <= len(prefix["input_ids"]):
                raise DatasetValidationError(
                    f"Assistant completion is empty: sample_id={sample_id}"
                )
            labels = (
                [-100] * len(prefix["input_ids"])
                + full["input_ids"][len(prefix["input_ids"]):]
            )
            rows.append({
                "input_ids": full["input_ids"],
                "attention_mask": full["attention_mask"],
                "labels": labels,
            })

        batch = self.tokenizer.pad(
            [
                {
                    "input_ids": row["input_ids"],
                    "attention_mask": row["attention_mask"],
                }
                for row in rows
            ],
            return_tensors="pt",
        )
        max_batch_length = batch["input_ids"].shape[1]
        labels = [
            row["labels"] + [-100] * (max_batch_length - len(row["labels"]))
            for row in rows
        ]
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = AssistantOnlyCollator(tokenizer, MAX_LENGTH)
print(train_dataset, validation_dataset)

Dataset({
    features: ['behavior_themes', 'category', 'dataset_version', 'id', 'messages', 'normalized_record_sha256', 'normalized_user_sha256', 'record_sha256', 'sample_sha256', 'source', 'source_dataset', 'source_file', 'source_id', 'source_row', 'source_sample_sha256', 'user_sha256'],
    num_rows: 2545
}) Dataset({
    features: ['behavior_themes', 'category', 'dataset_version', 'id', 'messages', 'normalized_record_sha256', 'normalized_user_sha256', 'record_sha256', 'sample_sha256', 'source', 'source_dataset', 'source_file', 'source_id', 'source_row', 'source_sample_sha256', 'user_sha256'],
    num_rows: 223
})


In [12]:
# 11. Trainer configuration
from transformers import Trainer, TrainingArguments

training_parameters = inspect.signature(TrainingArguments).parameters
required_parameters = {"optim", "gradient_checkpointing", "gradient_checkpointing_kwargs"}
missing_parameters = required_parameters - set(training_parameters)
if missing_parameters:
    raise RuntimeError(f"Transformers API lacks required controls: {sorted(missing_parameters)}")

trainer_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / ".trainer"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim=OPTIMIZER,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=BF16_SUPPORTED,
    fp16=not BF16_SUPPORTED,
    eval_strategy="epoch",
    save_strategy="no",
    report_to=[],
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=trainer_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
)

In [13]:
# 12. Final preflight — the next cell is the only training cell
EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print("Training mode:", TRAINING_MODE)
print("Adapter:", ADAPTER_NAME)
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Base:", BASE_MODEL)
print("Revision:", RESOLVED_BASE_REVISION)
print("Quantization: 4-bit NF4, double quantization=True")
print(f"LoRA: r={QLORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print("Targets:", TARGET_MODULES)
print("Optimizer:", OPTIMIZER)
print("Gradient checkpointing: True")
print("use_cache:", getattr(model.config, "use_cache", None))
print("MAX_LENGTH:", MAX_LENGTH)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
EXPECTED_OPTIMIZER_STEPS = math.ceil(
    len(train_records) / GRADIENT_ACCUMULATION_STEPS
)
print("Epochs:", NUM_TRAIN_EPOCHS)
print("Expected optimizer steps (approx):", EXPECTED_OPTIMIZER_STEPS)
assert NUM_TRAIN_EPOCHS == 1.0
assert TRAINING_MODE == "common_only"
assert ADAPTER_NAME == "hgfinance-common-v1"
assert len(train_dataset) == 2545
assert len(validation_dataset) == 223
assert OPTIMIZER == "paged_adamw_8bit"
assert getattr(model.config, "use_cache", None) is False

Training mode: common_only
Adapter: hgfinance-common-v1
Train: 2545
Validation: 223
Base: Qwen/Qwen2.5-14B-Instruct
Revision: cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8
Quantization: 4-bit NF4, double quantization=True
LoRA: r=16, alpha=32, dropout=0.05
Targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
Optimizer: paged_adamw_8bit
Gradient checkpointing: True
use_cache: False
MAX_LENGTH: 3072
Effective batch size: 16


In [15]:
# 13. TRAIN — intentionally separate and visually obvious
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# 14. Evaluate after training
eval_metrics = trainer.evaluate()
print(json.dumps(eval_metrics, indent=2, default=str))

In [ ]:
# 15. Save adapter-only artifacts and reproducibility metadata
import importlib.metadata as importlib_metadata

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")

def git_commit() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "UNKNOWN"

def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "UNKNOWN"

prepared_train_sha256 = sha256_file(PREPARED_TRAIN_PATH)
prepared_validation_sha256 = sha256_file(PREPARED_VALIDATION_PATH)
metadata = {
    "project": "HgFinance",
    "training_mode": "common_only",
    "adapter_name": ADAPTER_NAME,
    "adapter_version": ADAPTER_VERSION,
    "base_model": BASE_MODEL,
    "requested_base_revision": REQUESTED_BASE_REVISION,
    "resolved_base_revision": RESOLVED_BASE_REVISION,
    "common_train_sha256": EXPECTED_COMMON_TRAIN_SHA256,
    "common_validation_sha256": EXPECTED_COMMON_VALIDATION_SHA256,
    "prepared_train_sha256": prepared_train_sha256,
    "prepared_validation_sha256": prepared_validation_sha256,
    "train_count": len(train_dataset),
    "validation_count": len(validation_dataset),
    "benchmark_contamination": BENCHMARK_CONTAMINATION,
    "held_out_benchmarks": ["External-50", "Internal-v1", "Internal-v2 / EmployeeReasoning"],
    "lora": {
        "r": QLORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "max_production_rank": 32,
    },
    "quantization": {
        "load_in_4bit": True,
        "quant_type": "nf4",
        "double_quant": True,
        "compute_dtype": str(COMPUTE_DTYPE),
    },
    "optimizer": OPTIMIZER,
    "max_length": MAX_LENGTH,
    "training_args": {
        "epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
        "gradient_checkpointing": True,
        "gradient_checkpointing_use_reentrant": False,
        "use_cache": False,
        "eval_strategy": "epoch",
        "save_strategy": "no",
    },
    "runtime": {
        "gpu": GPU_NAME,
        "vram_gib": VRAM_GIB,
        "torch": package_version("torch"),
        "transformers": package_version("transformers"),
        "peft": package_version("peft"),
        "accelerate": package_version("accelerate"),
        "bitsandbytes": package_version("bitsandbytes"),
        "datasets": package_version("datasets"),
    },
    "git_commit": git_commit(),
    "token_length_audit": TOKEN_LENGTH_AUDIT,
    "final_eval_metrics": eval_metrics,
}
(OUTPUT_DIR / "training_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, default=str) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "adapter": ADAPTER_NAME,
    "adapter_model": str(OUTPUT_DIR / "adapter_model.safetensors"),
    "metadata": str(OUTPUT_DIR / "training_metadata.json"),
}, indent=2))

In [ ]:
# 16. Adapter-only artifact verification
required_files = [
    OUTPUT_DIR / "adapter_model.safetensors",
    OUTPUT_DIR / "adapter_config.json",
    OUTPUT_DIR / "training_metadata.json",
]
for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)

for forbidden_name in ("pytorch_model.bin", "model.safetensors"):
    if (OUTPUT_DIR / forbidden_name).exists():
        raise RuntimeError(f"Full base-model artifact found: {forbidden_name}")

saved_metadata = json.loads((OUTPUT_DIR / "training_metadata.json").read_text(encoding="utf-8"))
assert saved_metadata["training_mode"] == "common_only"
assert saved_metadata["adapter_name"] == ADAPTER_NAME
assert saved_metadata["train_count"] == 2545
assert saved_metadata["validation_count"] == 223
assert saved_metadata["resolved_base_revision"] == RESOLVED_BASE_REVISION
print("Adapter-only verification: PASS")
print("Output:", OUTPUT_DIR)

In [ ]:
# 17. Small deterministic sanity generation — not a formal benchmark
model.eval()

def pick_record(keyword: str, fallback_index: int) -> dict:
    for record in validation_records:
        text = " ".join(message["content"] for message in record["messages"]).lower()
        if keyword in text:
            return record
    return validation_records[fallback_index]

smoke_records = [
    pick_record("evidence", 0),
    pick_record("insufficient", 1),
    pick_record("risk", 2),
]

for record in smoke_records:
    prompt_messages = record["messages"][:-1]
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prediction = tokenizer.decode(
        generated[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )
    print("\nID:", record["id"])
    print("Question:", next(message["content"] for message in prompt_messages if message["role"] == "user"))
    print("Gold:", record["messages"][-1]["content"])
    print("Prediction:", prediction)

In [ ]:
# 18. ZIP adapter artifacts and optional Drive save
ZIP_STAGING = OUTPUT_DIR / ".adapter_zip_staging"
if ZIP_STAGING.exists():
    shutil.rmtree(ZIP_STAGING)
ZIP_STAGING.mkdir(parents=True)

for filename in ("adapter_model.safetensors", "adapter_config.json", "training_metadata.json"):
    shutil.copy2(OUTPUT_DIR / filename, ZIP_STAGING / filename)
if (OUTPUT_DIR / "tokenizer").is_dir():
    shutil.copytree(OUTPUT_DIR / "tokenizer", ZIP_STAGING / "tokenizer")

ZIP_PATH = REPO_ROOT / "hgfinance-common-v1.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=ZIP_STAGING,
)
shutil.rmtree(ZIP_STAGING)

print("ZIP:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MiB")

USE_GOOGLE_DRIVE = False
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/HgFinance/adapters")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, DRIVE_SAVE_DIR / ZIP_PATH.name)
    print("Drive copy:", DRIVE_SAVE_DIR / ZIP_PATH.name)

## Post-training separation

This notebook only produces the shared Common behavior adapter
hgfinance-common-v1.

Formal External-50, Internal-v1, and Internal-v2 / EmployeeReasoning
evaluation remains outside this notebook. The adapter must be evaluated against
the production AWQ serving base only after adapter artifacts and metadata are
reviewed.

Do not merge the adapter into the 14B base model, do not train the production
AWQ checkpoint, and do not treat sanity generation as a promotion result.